# **Trabajo Integrador**: “Clasificador de Imagenes Perros y Gatos”


 **Departamento de Informática: Asignatura Inteligencia Artificial**


**Alumnos:** Bacigaluppe Maximiliano - Burguener Rocio
        


**Profesora:**  Dra. Sonia I. Mariño -  Lic. Jaquelina Escalante


**Año: 2023**



In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

#Descarga del set de datos
datos, metadatos = tfds.load('cats_vs_dogs', as_supervised=True, with_info=True)

Muestra una cuadrícula de subplots que contiene las primeras 25 imágenes en escala de grises

In [ ]:
import matplotlib.pyplot as plt
import cv2

plt.figure(figsize=(20, 20))

tamanio_img=100

for i, (imagen, etiqueta) in enumerate(datos['train'].take(25)):
    imagen = cv2.resize(imagen.numpy(), (tamanio_img, tamanio_img))
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(imagen)
    plt.imshow(imagen, cmap='gray')

In [ ]:
datos_entrenamiento = []

Procesa todos los datos del conjunto de datos 'train', redimensiona las imágenes a un tamaño de 100x100 píxeles, las convierte a escala de grises y las agrega junto con sus etiquetas a una lista llamada datos_entrenamiento.

In [ ]:
for i, (imagen, etiqueta) in enumerate(datos['train']): #Todos los datos
    imagen = cv2.resize(imagen.numpy(), (tamanio_img, tamanio_img))
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    imagen = imagen.reshape(tamanio_img, tamanio_img, 1) #Cambiar tamaño a 100, 100, 1
    datos_entrenamiento.append([imagen, etiqueta])


In [ ]:
len(datos_entrenamiento)

In [ ]:
x = [] #imagen de entrada (pixeles)
y = [] #etiqueta (perro o gato)

for imagen, etiqueta in datos_entrenamiento:
  x.append(imagen)
  y.append(etiqueta)

Convierte la lista de imágenes "x" en un array NumPy, lo convierte a tipo de datos float y luego normaliza los valores de los píxeles dividiendo por 255, de modo que los valores estén en el rango de 0 a 1.

In [ ]:

import numpy as np

x = np.array(x).astype(float) / 255

In [ ]:
y = np.array(y)

El generador de imágenes (ImageDataGenerator) se utiliza para aplicar técnicas de aumentación de datos durante el entrenamiento de un modelo de aprendizaje automático. La aumentación de datos implica realizar transformaciones aleatorias en las imágenes de entrada con el objetivo de aumentar la variabilidad del conjunto de datos y mejorar la capacidad del modelo para generalizar y evitar el sobreajuste.

In [ ]:
#Importar el generador con aumento de datos

from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=15,
    zoom_range=[0.7, 1.4],
    horizontal_flip=True,
    vertical_flip=True
)

datagen.fit(x)

plt.figure(figsize=(20, 8))

for imagen, etiqueta in datagen.flow(x, y, batch_size=10, shuffle=False):
  for i in range(10):
    plt.subplot(2,5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(imagen[i].reshape(100, 100), cmap="gray")
  break

In [ ]:
#Entrenamiento de 3 modelo con aumento de datos
modeloDenso_AD = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(100, 100, 1)),#Capa de entrada recibe los 10.000 pixeles
    tf.keras.layers.Dense(150, activation='relu'), #Capa densa con 150 neuronas
    tf.keras.layers.Dense(150, activation='relu'), #Capa densa con 150 nueronas
    tf.keras.layers.Dense(1, activation='sigmoid') #Capa de salida con una salida
])

#Posee 3 capas de convolucionales con 32, 64 y 128 filtros y una capa densa de 100 neuronas y una capa de salida
modeloCNN_AD = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

#Se agrega un Dropout de 0.5 antes de la capa densa la cual posee 250 neuronas
modeloCNN2_AD = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(250, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

In [ ]:
#Se complila el modelo con el optimizador "adam", funcion de perdida binary_crossentropy (que se aplica para resultados binarios) y las metricas de precision

modeloDenso_AD.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001), # Optimizador "adam", proporciona una tasa de aprendizaje adaptable y eficiencia.
                      loss='binary_crossentropy', #perdida
                       metrics=['accuracy']) #presición

modeloCNN_AD.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                       metrics=['accuracy'])

modeloCNN2_AD.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
                      loss='binary_crossentropy',
                       metrics=['accuracy'])

In [ ]:
#Cuanto datos de entranamiento y cuanto de validacion

len(x) * .85 #19700
len(x) - 19700

x_entrenamiento = x[:19700] #85% de entrenamiento
x_validacion = x[19700:]    #15% de prueba

y_entrenamiento = y[:19700]
y_validacion = y[19700:]

In [ ]:
#genedar de dato para el entrenamiento
data_gen_entrenamiento = datagen.flow(x_entrenamiento, y_entrenamiento, batch_size=32)

In [ ]:
from keras.callbacks import TensorBoard

#Entrena el modelo y lo adjunta a una archivo "denso_AD"

tensorboarDenso_AD = TensorBoard(log_dir='logs/denso_AD')

modeloDenso_AD.fit(
    data_gen_entrenamiento,
    epochs=100, batch_size=32,
    validation_data=(x_validacion, y_validacion), #Datos de validacion (x=imagenes; y=etiquetas)
    steps_per_epoch=int(np.ceil(len(x_entrenamiento) / float(32))), #Variables para el entrenamiento
    validation_steps=int(np.ceil(len(x_entrenamiento) / float(32))), #Variable para la validacion
    callbacks=[tensorboarDenso_AD]
)

In [ ]:
#Entrena el modelo y lo adjunta a una archivo "cnn_AD"

tensorboarCNN_AD = TensorBoard(log_dir='logs/cnn_AD')

modeloCNN_AD.fit(
    data_gen_entrenamiento,
    epochs=100, batch_size=32,
    validation_data=(x_validacion, y_validacion), #Datos de validacion (x=imagenes; y=etiquetas)
    steps_per_epoch=int(np.ceil(len(x_entrenamiento) / float(32))), #Variables para el entrenamiento
    validation_steps=int(np.ceil(len(x_entrenamiento) / float(32))), #Variable para la validacion
    callbacks=[tensorboarCNN_AD]
)

In [ ]:
#Entrena el modelo y lo adjunta a una archivo "cnn2_AD"

tensorboarCNN2_AD = TensorBoard(log_dir='logs/cnn2_AD')

modeloCNN2_AD.fit(
    data_gen_entrenamiento,
    epochs=100, batch_size=32,
    validation_data=(x_validacion, y_validacion), #Datos de validacion (x=imagenes; y=etiquetas)
    steps_per_epoch=int(np.ceil(len(x_entrenamiento) / float(32))), #Variables para el entrenamiento
    validation_steps=int(np.ceil(len(x_entrenamiento) / float(32))), #Variable para la validacion
    callbacks=[tensorboarCNN2_AD]
)

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/

In [ ]:
import gc

gc.collect()

In [ ]:
modeloCNN_AD.save('perros-gatos-cnn-ad.h5')

In [ ]:
!pip install tensorflowjs

In [ ]:
!mkdir carpeta_salida

In [ ]:
!tensorflowjs_converter --input_format keras perros-gatos-cnn-ad.h5 carpeta_salida